# POMDPs and Belief: Inferring a Hidden World

Companion notebook for Tutorial 3, Chapter 24. Runs every code block from the chapter.

Chapter: https://josephausterweil.github.io/probintro/intro2/24_pomdps_belief_inference/

In [ ]:
# Colab setup
!pip -q install "genjax==0.10.3" "jax==0.5.3" 2>/dev/null

In [ ]:
import jax.numpy as jnp
from genjax import gen, categorical, ChoiceMap

# states: 0 = tiger-LEFT, 1 = tiger-RIGHT ; observations: 0 = hear-LEFT, 1 = hear-RIGHT
ACC = 0.85                                          # listening is 85% accurate
OBS = jnp.array([[ACC, 1 - ACC],                    # tiger-left  -> hear-left .85 / right .15
                 [1 - ACC, ACC]])                   # tiger-right -> hear-left .15 / right .85
R_LISTEN, R_CORRECT, R_TIGER = -1.0, 10.0, -100.0   # listen -1, open correct +10, open tiger -100
HEAR_LEFT, HEAR_RIGHT = 0, 1

@gen
def listen(belief):                                 # the generative model of one listen
    s = categorical(jnp.log(belief)) @ "s"          # the hidden state, drawn from the belief
    categorical(jnp.log(OBS[s])) @ "o"              # the growl we actually hear

def update_belief(belief, obs):                     # b'(s) proportional to P(obs|s) b(s)
    logp = jnp.array([listen.assess(ChoiceMap.d({"s": s, "o": int(obs)}), (belief,))[0]
                      for s in range(2)])
    p = jnp.exp(logp - jnp.max(logp))
    return p / jnp.sum(p)

def E_open_right(belief):                            # expected reward of opening the right door
    bL, bR = float(belief[0]), float(belief[1])
    return bL * R_CORRECT + bR * R_TIGER            # +10 if tiger-left, -100 if tiger-right

In [ ]:
b = jnp.array([0.5, 0.5])                            # uniform prior: no idea which door
print(f"start: P(tiger-left) = {float(b[0]):.4f}")
for k in range(1, 4):
    b = update_belief(b, HEAR_LEFT)                 # the tiger keeps sounding from the left
    print(f"  after {k} left-growl(s): P(tiger-left) = {float(b[0]):.4f}   E[open-right] = {E_open_right(b):+.2f}")

In [ ]:
# E[open-right] is linear in the belief b = P(tiger-left):  10*b - 100*(1-b) = 110*b - 100.
# It beats listening (-1) when 110*b - 100 > -1, i.e. b > 99/110.
threshold = 99 / 110
print(f"E[open-right] = 110*b - 100   (a straight line in b)")
print(f"opening beats listening when b > {threshold:.4f}")
b1 = update_belief(jnp.array([0.5, 0.5]), HEAR_LEFT)
b2 = update_belief(b1, HEAR_LEFT)
print(f"  1 growl : b = {float(b1[0]):.3f}  ->  below {threshold:.2f}, LISTEN again")
print(f"  2 growls: b = {float(b2[0]):.3f}  ->  above {threshold:.2f}, OPEN the right door")

In [ ]:
b_cancel = update_belief(update_belief(jnp.array([0.5, 0.5]), HEAR_LEFT), HEAR_RIGHT)
print(f"hear-left then hear-right: P(tiger-left) = {float(b_cancel[0]):.4f}")

In [ ]:
NROWS, NCOLS, NA = 3, 3, 4
NS = NROWS * NCOLS
START = 1                                           # (0,1) bottom-middle
GOALS = {"left": 6, "right": 8}                     # top-left / top-right
GNAMES = list(GOALS); GOAL_STATES = jnp.array([GOALS[g] for g in GNAMES])
GAMMA, BETA = 0.9, 3.0
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3

def step(s, a):
    r, c = s // NCOLS, s % NCOLS
    r = jnp.clip(jnp.where(a == UP, r + 1, jnp.where(a == DOWN, r - 1, r)), 0, NROWS - 1)
    c = jnp.clip(jnp.where(a == LEFT, c - 1, jnp.where(a == RIGHT, c + 1, c)), 0, NCOLS - 1)
    return r * NCOLS + c

TRANS = jnp.array([[int(step(s, a)) for a in range(NA)] for s in range(NS)])

def q_for_goal(goal):
    reward = (TRANS == goal).astype(jnp.float32)
    is_goal = (jnp.arange(NS) == goal)
    V = jnp.zeros(NS)
    for _ in range(50):
        V = jnp.where(is_goal, 0.0, jnp.max(reward + GAMMA * V[TRANS], axis=1))
    return reward + GAMMA * V[TRANS]

LOGITS = BETA * jnp.stack([q_for_goal(g) for g in GOAL_STATES])
GPRIOR = jnp.ones(len(GNAMES)) / len(GNAMES)

@gen
def watcher(states):
    g = categorical(jnp.log(GPRIOR)) @ "goal"
    for t in range(len(states)):
        categorical(LOGITS[g, states[t]]) @ f"a_{t}"

def posterior_true(true_idx, states, actions):      # observer's prob on the TRUE goal
    logp = jnp.array([watcher.assess(
        ChoiceMap.d({"goal": g, **{f"a_{t}": int(actions[t]) for t in range(len(actions))}}),
        (jnp.asarray(states),))[0] for g in range(len(GNAMES))])
    post = jnp.exp(logp - jnp.max(logp)); post = post / jnp.sum(post)
    return float(post[true_idx])

def states_of(start, acts):
    s, out = start, []
    for a in acts: out.append(int(s)); s = int(step(jnp.array(s), jnp.array(a)))
    return out

In [ ]:
TRUE = GNAMES.index("right")
doing   = [UP, UP, RIGHT]        # straight up the middle, then over: ambiguous early
showing = [RIGHT, UP, UP]        # commit to the right immediately: legible
print("observer's P(true goal = top-right) after each step:")
print("  step   doing(efficient)   showing(legible)")
for k in range(1, 4):
    pd = posterior_true(TRUE, states_of(START, doing)[:k],   doing[:k])
    ps = posterior_true(TRUE, states_of(START, showing)[:k], showing[:k])
    print(f"   {k}          {pd:.3f}              {ps:.3f}")